# Late Fusion — Image + Text + Clinical Features

Combines three independent branches via concatenation:
- **Image**: frozen ViT-B/16 CLS token [768] — baseline checkpoint from Component 1
- **Text**: frozen BioClinicalBERT CLS token [768] — impression + findings when available
- **Clinical**: 14 encoded features from notebook 07

**Key design**: both encoders are frozen → pre-compute all embeddings once and cache
as numpy arrays. Training then only updates the small fusion head (no image loading,
no transformer forward passes per step — as fast as the clinical MLP).

**Pre-computation** (~35 min on MacBook MPS, ~10 min on Colab A100):
run once, embeddings saved to `checkpoints/embeddings/`. Skip if files exist.

**Fusion head**: concat([768, 768, 14]) = [1550] → Linear(256) → ReLU → Dropout → Linear(14)

**Outputs**
- `checkpoints/embeddings/{train,valid}_{img,txt}_emb.npy`
- `checkpoints/fusion_late_best.pt`
- `checkpoints/fusion_late_auc.csv`
- `maps/fusion_late_summary.png`

In [ ]:
# ── Cell 1 — Environment ──────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/dizertatie_project'):
        !git clone https://github.com/daviDpaD18/diz.git /content/diz
        os.symlink('/content/diz/dizertatie_project', '/content/dizertatie_project')
    sys.path.insert(0, '/content/dizertatie_project')
    if not os.path.exists('/content/train'):
        DRIVE = '/content/drive/MyDrive/dizertatie'
        !cp {DRIVE}/train.zip /content/train.zip
        !unzip -q /content/train.zip -d /content/ && rm /content/train.zip
    if not os.path.exists('/content/valid'):
        !cp -r {DRIVE}/valid /content/valid
    import importlib, src.config; importlib.reload(src.config)
except ImportError:
    IS_COLAB = False
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

print('Colab:', IS_COLAB)

In [ ]:
# ── Cell 2 — Imports + directories ───────────────────────────────────────────
!pip install -q transformers

import json, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from src.config import IMAGE_ROOT, SPLITS_DIR, WEIGHTS_DIR, CKPT_DIR
from src.dataset import LABEL_COLS, CheXpertDataset, eval_transforms, remap_uncertain

EMB_DIR  = CKPT_DIR / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR = (CKPT_DIR.parent / 'maps') if IS_COLAB \
           else Path('..').resolve() / 'maps'
MAPS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cpu')
)
EMB_BATCH = 128 if torch.cuda.is_available() else 32
print(f'Device: {DEVICE}  EMB_BATCH: {EMB_BATCH}')

In [ ]:
# ── Cell 3 — Pre-compute ViT image embeddings ─────────────────────────────────
#
# Loads the best baseline checkpoint from Component 1.
# Registers a forward hook on model.norm to capture the CLS token [B, 768]
# before the classification head. Runs eval_transforms (deterministic).
# Saves: {train,valid}_img_emb.npy  shape [N, 768]  float32
# Skips if both files already exist.

seeds = [42, 123, 456]
BASELINE_CKPT = max(
    (CKPT_DIR / f'baseline_seed{s}_best.pt' for s in seeds),
    key=lambda p: torch.load(p, map_location='cpu',
                             weights_only=False)['val_macro_auc']
)
print(f'Baseline checkpoint: {BASELINE_CKPT.name}')

TRAIN_CSV = SPLITS_DIR / ('train_full_clin.csv' if IS_COLAB else 'train_full_clin.csv')
VALID_CSV = SPLITS_DIR / 'valid_frontal_clin.csv'

TRAIN_IMG = EMB_DIR / 'train_img_emb.npy'
VALID_IMG = EMB_DIR / 'valid_img_emb.npy'

if TRAIN_IMG.exists() and VALID_IMG.exists():
    print('Image embeddings already exist — skipping.')
else:
    base_ckpt  = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
    vit        = timm.create_model(base_ckpt['hparams']['model'],
                                   pretrained=False, num_classes=14)
    vit.load_state_dict(base_ckpt['model_state_dict'])
    vit.eval().to(DEVICE)

    # Hook: capture CLS token output from the final norm layer
    _cls_buf = []
    def _hook(module, inp, out):
        _cls_buf.append(out[:, 0, :].detach().cpu().float())
    handle = vit.norm.register_forward_hook(_hook)

    for csv_path, save_path, split_name in [
        (TRAIN_CSV, TRAIN_IMG, 'train'),
        (VALID_CSV, VALID_IMG, 'valid'),
    ]:
        df  = pd.read_csv(csv_path)
        ds  = CheXpertDataset(df, IMAGE_ROOT, eval_transforms)
        ldr = DataLoader(ds, batch_size=EMB_BATCH, shuffle=False,
                         num_workers=4 if IS_COLAB else 0, pin_memory=IS_COLAB)
        _cls_buf.clear()
        with torch.no_grad():
            for imgs, *_ in tqdm(ldr, desc=f'ViT embeddings [{split_name}]'):
                vit(imgs.to(DEVICE))
        emb = torch.cat(_cls_buf).numpy()   # [N, 768]
        np.save(save_path, emb)
        print(f'  Saved {save_path.name}: {emb.shape}')

    handle.remove()
    del vit
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── Cell 4 — Pre-compute BioClinicalBERT text embeddings ──────────────────────
#
# Text = section_impression (99.9% coverage) +  section_findings when available.
# Format: "impression [SEP] findings" or just "impression".
# Rows with no text → zero vector (217 train rows, 0 val rows).
# Saves: {train,valid}_txt_emb.npy  shape [N, 768]  float32
# Skips if both files already exist.

BERT_NAME  = 'emilyalsentzer/Bio_ClinicalBERT'
TRAIN_TXT  = EMB_DIR / 'train_txt_emb.npy'
VALID_TXT  = EMB_DIR / 'valid_txt_emb.npy'
TXT_BATCH  = 64 if torch.cuda.is_available() else 32
MAX_LEN    = 256   # p95 of impression = 81 words; 256 tokens is safe and faster than 512


def build_texts(df):
    """Combine impression + findings. Return list of strings; empty string = no text."""
    texts = []
    for _, row in df.iterrows():
        imp = str(row.get('section_impression') or '').strip()
        fin = str(row.get('section_findings')   or '').strip()
        if imp and fin:
            texts.append(imp + ' ' + fin)
        elif imp:
            texts.append(imp)
        elif fin:
            texts.append(fin)
        else:
            texts.append('')   # zero vector will be used
    return texts


if TRAIN_TXT.exists() and VALID_TXT.exists():
    print('Text embeddings already exist — skipping.')
else:
    print(f'Loading {BERT_NAME}...')
    tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
    bert      = AutoModel.from_pretrained(BERT_NAME).eval().to(DEVICE)

    for csv_path, save_path, split_name in [
        (TRAIN_CSV, TRAIN_TXT, 'train'),
        (VALID_CSV, VALID_TXT, 'valid'),
    ]:
        df    = pd.read_csv(csv_path, low_memory=False)
        texts = build_texts(df)
        N     = len(texts)
        emb   = np.zeros((N, 768), dtype=np.float32)

        for start in tqdm(range(0, N, TXT_BATCH),
                          desc=f'BERT embeddings [{split_name}]'):
            batch_texts = texts[start:start + TXT_BATCH]
            # Separate empty texts — use zero vector, no GPU waste
            non_empty   = [(i, t) for i, t in enumerate(batch_texts) if t]
            if not non_empty:
                continue
            idxs, valid_texts = zip(*non_empty)
            enc = tokenizer(list(valid_texts), return_tensors='pt',
                            max_length=MAX_LEN, padding=True,
                            truncation=True).to(DEVICE)
            with torch.no_grad():
                out = bert(**enc)
            cls = out.last_hidden_state[:, 0, :].cpu().float().numpy()
            for local_i, global_i in enumerate(idxs):
                emb[start + global_i] = cls[local_i]

        np.save(save_path, emb)
        n_zero = (emb == 0).all(axis=1).sum()
        print(f'  Saved {save_path.name}: {emb.shape}  zero rows: {n_zero}')

    del bert, tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ── Cell 5 — Load embeddings + clinical features ──────────────────────────────

import sys
sys.path.insert(0, str(Path('..').resolve()))

# Reuse encode_features from notebook 07 via the saved encoder
with open(WEIGHTS_DIR / 'clinical_encoder.pkl', 'rb') as f:
    enc_data = pickle.load(f)
clin_stats   = enc_data['stats']
FEATURE_NAMES = enc_data['feature_names']
N_CLIN        = len(FEATURE_NAMES)

# Redefine encode_features (same logic as notebook 07, reuse stats)
def encode_features(df, stats):
    feats = pd.DataFrame(index=df.index)
    age   = df['Age'].fillna(stats['age_median'])
    feats['age']         = (age - stats['age_mean']) / (stats['age_std'] + 1e-8)
    feats['sex_male']    = (df['Sex'] == 'Male').astype(float)
    feats['sex_unknown'] = (~df['Sex'].isin(['Male', 'Female'])).astype(float)
    feats['ap_pa']       = (df['AP/PA'] == 'AP').astype(float)
    race = df.get('race', pd.Series('Unknown', index=df.index)).fillna('Unknown')
    feats['race_black']  = (race == 'Black').astype(float)
    feats['race_asian']  = (race == 'Asian').astype(float)
    feats['race_other']  = race.isin(['Other','Unknown','Pacific Islander','Native American']).astype(float)
    eth = df.get('ethnicity', pd.Series('', index=df.index)).fillna('')
    feats['hispanic']    = eth.str.contains('Hispanic/Latino', na=False).astype(float)
    ins = df.get('insurance_type', pd.Series('Unknown', index=df.index)).fillna('Unknown')
    feats['ins_medicare'] = (ins == 'Medicare').astype(float)
    feats['ins_medicaid'] = (ins == 'Medicaid').astype(float)
    feats['ins_unknown']  = (ins == 'Unknown').astype(float)
    feats['ins_other']    = (ins == 'Other').astype(float)
    bmi_raw     = df.get('recent_bmi', pd.Series(np.nan, index=df.index))
    bmi_missing = bmi_raw.isna()
    bmi         = bmi_raw.fillna(stats['bmi_median'])
    feats['bmi']         = (bmi - stats['bmi_median']) / (stats['bmi_std'] + 1e-8)
    feats['bmi_missing'] = bmi_missing.astype(float)
    return feats[FEATURE_NAMES].values.astype(np.float32)

df_train = pd.read_csv(TRAIN_CSV, low_memory=False)
df_valid = pd.read_csv(VALID_CSV, low_memory=False)

X_clin_train = encode_features(df_train, clin_stats)   # [N_train, 14]
X_clin_valid = encode_features(df_valid, clin_stats)   # [202, 14]

img_train = np.load(TRAIN_IMG)   # [N_train, 768]
img_valid = np.load(VALID_IMG)   # [202, 768]
txt_train = np.load(TRAIN_TXT)   # [N_train, 768]
txt_valid = np.load(VALID_TXT)   # [202, 768]

print(f'Train — img: {img_train.shape}  txt: {txt_train.shape}  clin: {X_clin_train.shape}')
print(f'Val   — img: {img_valid.shape}  txt: {txt_valid.shape}  clin: {X_clin_valid.shape}')

# Class weights
RARE_LABELS  = ['Pneumonia', 'Fracture', 'Lung Lesion', 'Pleural Other']
RELIABLE_MIN = 5
val_pos      = df_valid[LABEL_COLS].sum().to_dict()

with open(WEIGHTS_DIR / 'class_weights.json') as f:
    cw = json.load(f)
CLASS_WEIGHTS = torch.tensor([cw[l] for l in LABEL_COLS], dtype=torch.float32).to(DEVICE)

def weighted_bce(logits, labels):
    return (F.binary_cross_entropy_with_logits(
        logits, labels, reduction='none') * CLASS_WEIGHTS).mean()

In [ ]:
# ── Cell 6 — Fusion Dataset + model ──────────────────────────────────────────

IMG_DIM   = 768
TXT_DIM   = 768
FUSED_DIM = IMG_DIM + TXT_DIM + N_CLIN   # 1550


class FusionDataset(Dataset):
    """Returns (img_emb, txt_emb, clin_feats, labels) — no image loading."""

    def __init__(self, img_emb, txt_emb, clin_feats, df):
        self.img   = torch.tensor(img_emb,   dtype=torch.float32)
        self.txt   = torch.tensor(txt_emb,   dtype=torch.float32)
        self.clin  = torch.tensor(clin_feats,dtype=torch.float32)
        labels     = remap_uncertain(df, LABEL_COLS)[LABEL_COLS].values.astype(float)
        self.labels = torch.tensor(labels,   dtype=torch.float32)

    def __len__(self):         return len(self.img)
    def __getitem__(self, i):  return self.img[i], self.txt[i], self.clin[i], self.labels[i]


class LateFusionHead(nn.Module):
    """
    Concatenates frozen image, text, and clinical embeddings,
    then classifies via a two-layer MLP.
    Only this module's parameters are trained.
    """

    def __init__(self, fused_dim: int = FUSED_DIM, num_classes: int = 14):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, img, txt, clin):
        x = torch.cat([img, txt, clin], dim=-1)   # [B, 1550]
        return self.net(x)


TRAIN_EPOCHS = 50
BATCH_SIZE   = 512
LR           = 3e-4
WEIGHT_DECAY = 1e-2

train_ds = FusionDataset(img_train, txt_train, X_clin_train, df_train)
val_ds   = FusionDataset(img_valid, txt_valid, X_clin_valid, df_valid)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = LateFusionHead().to(DEVICE)
print(f'Fusion head parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Train: {len(train_ds):,}  Val: {len(val_ds)}')

In [ ]:
# ── Cell 7 — Training ─────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, img_scale=1.0, txt_scale=1.0, clin_scale=1.0):
    """Evaluate with optional branch zeroing for ablation."""
    model.eval()
    all_probs, all_labels = [], []
    for img, txt, clin, labels in val_loader:
        img  = img.to(DEVICE)  * img_scale
        txt  = txt.to(DEVICE)  * txt_scale
        clin = clin.to(DEVICE) * clin_scale
        all_probs.append(torch.sigmoid(model(img, txt, clin)).cpu())
        all_labels.append(labels)
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = {}
    for i, label in enumerate(LABEL_COLS):
        auc[label] = roc_auc_score(labels[:, i], probs[:, i]) \
                     if labels[:, i].sum() > 0 else float('nan')
    return float(np.nanmean(list(auc.values()))), auc


optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = TRAIN_EPOCHS * len(train_loader)
scheduler   = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.05 * total_steps),
    num_training_steps=total_steps)

best_auc  = 0.0
save_path = CKPT_DIR / 'fusion_late_best.pt'

for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    running = 0.0
    for img, txt, clin, labels in train_loader:
        img, txt, clin, labels = (img.to(DEVICE), txt.to(DEVICE),
                                   clin.to(DEVICE), labels.to(DEVICE))
        optimizer.zero_grad()
        loss = weighted_bce(model(img, txt, clin), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item()

    macro, auc_per_label = evaluate(model)
    flag = ' ← best' if macro > best_auc else ''
    if epoch % 5 == 0 or flag:
        print(f'Epoch {epoch:>3} | loss {running/len(train_loader):.4f} '
              f'| AUC {macro:.4f}{flag}')

    if macro > best_auc:
        best_auc = macro
        torch.save({
            'epoch':             epoch,
            'model_state_dict':  model.state_dict(),
            'val_macro_auc':     macro,
            'val_auc_per_label': auc_per_label,
        }, save_path)

print(f'\nBest macro AUC: {best_auc:.4f}')

In [ ]:
# ── Cell 8 — AUC comparison table ────────────────────────────────────────────
#
# Compares: image-only baseline | clinical-only | late fusion
# Loads saved checkpoints from Component 1 (baseline) and notebook 07 (clinical).

model.load_state_dict(
    torch.load(save_path, map_location='cpu', weights_only=False)['model_state_dict'])

_, fusion_auc = evaluate(model)

baseline_ckpt = torch.load(
    max((CKPT_DIR / f'baseline_seed{s}_best.pt' for s in [42, 123, 456]),
        key=lambda p: torch.load(p, map_location='cpu',
                                 weights_only=False)['val_macro_auc']),
    map_location='cpu', weights_only=False)

clinical_ckpt = torch.load(CKPT_DIR / 'clinical_mlp_best.pt',
                            map_location='cpu', weights_only=False)

rows = []
for label in LABEL_COLS:
    n_pos = int(val_pos.get(label, 0))
    base  = baseline_ckpt['val_auc_per_label'].get(label, float('nan'))
    clin  = clinical_ckpt['val_auc_per_label'].get(label, float('nan'))
    fuse  = fusion_auc.get(label, float('nan'))
    rows.append({
        'Label':      label,
        'Val+':       n_pos,
        'Rare':       '★' if label in RARE_LABELS else '',
        'image_only': base,
        'clinical':   clin,
        'late_fusion':fuse,
        'Δ_vs_image': fuse - base if not (np.isnan(fuse) or np.isnan(base)) else float('nan'),
    })

df_res = pd.DataFrame(rows).set_index('Label')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(df_res.round(4).to_string())

print('\nMacro AUC (reliable labels, Val+ ≥ 5):')
for name, col in [('image_only', 'image_only'),
                  ('clinical',   'clinical'),
                  ('late_fusion','late_fusion')]:
    vals = [df_res.loc[l, col] for l in LABEL_COLS
            if int(val_pos.get(l, 0)) >= RELIABLE_MIN
            and not np.isnan(df_res.loc[l, col])]
    with warnings.catch_warnings(): warnings.simplefilter('ignore')
    print(f'  {name:<14}: {np.nanmean(vals):.4f}')

df_res.to_csv(CKPT_DIR / 'fusion_late_auc.csv')
print('\nSaved fusion_late_auc.csv')

In [ ]:
# ── Cell 9 — Ablation: contribution of each branch ───────────────────────────
#
# Zero out each branch in turn to measure its individual contribution.
# This works because the fusion head was trained on real embeddings;
# zeroing a branch simulates its absence at inference.
#
# Configurations:
#   full         = image + text + clinical
#   no_image     = zero image  (text + clinical only)
#   no_text      = zero text   (image + clinical only)
#   no_clinical  = zero clinical (image + text only)
#   image_only   = zero text + zero clinical
#   text_only    = zero image + zero clinical

ablation_configs = {
    'full':          (1.0, 1.0, 1.0),
    'no_image':      (0.0, 1.0, 1.0),
    'no_text':       (1.0, 0.0, 1.0),
    'no_clinical':   (1.0, 1.0, 0.0),
    'image_only':    (1.0, 0.0, 0.0),
    'text_only':     (0.0, 1.0, 0.0),
    'clinical_only': (0.0, 0.0, 1.0),
}

ablation_results = {}
print(f'{"Config":<18} {"Macro AUC":>10}')
print('-' * 30)
for config, (img_s, txt_s, clin_s) in ablation_configs.items():
    macro, per_label = evaluate(model, img_s, txt_s, clin_s)
    ablation_results[config] = per_label
    print(f'{config:<18} {macro:>10.4f}')

# Save ablation per-label results
abl_rows = []
for label in LABEL_COLS:
    if int(val_pos.get(label, 0)) < RELIABLE_MIN:
        continue
    row = {'Label': label}
    for config in ablation_configs:
        row[config] = ablation_results[config].get(label, float('nan'))
    abl_rows.append(row)

df_abl = pd.DataFrame(abl_rows).set_index('Label')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('\nPer-label AUC by ablation config (reliable labels only):')
    print(df_abl.round(4).to_string())

df_abl.to_csv(CKPT_DIR / 'fusion_late_ablation.csv')
print('\nSaved fusion_late_ablation.csv')

In [ ]:
# ── Cell 10 — Full comparison table: all experiments to date ──────────────────
#
# Loads saved AUC CSVs from every completed notebook and builds one table.
# Columns: baseline | head_25/50/75 | dynvit_70/50/30 | ca_70/50/30 | clinical | fusion
# Rows: reliable labels only (Val+ ≥ 5).
# Macro AUC summary at the bottom.
#
# Gracefully skips any CSV that does not yet exist (warn and leave NaN).

def load_col(csv_path, index_col, col_name, alias=None):
    """Load one column from a CSV into a Series indexed by label. NaN if missing."""
    alias = alias or col_name
    if not Path(csv_path).exists():
        print(f'  [warn] {Path(csv_path).name} not found — {alias} will be NaN')
        return pd.Series(dtype=float, name=alias)
    df = pd.read_csv(csv_path, index_col=index_col)
    if col_name not in df.columns:
        print(f'  [warn] column {col_name!r} missing in {Path(csv_path).name}')
        return pd.Series(dtype=float, name=alias)
    return df[col_name].rename(alias)

rel = [l for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]

# ── collect all series ────────────────────────────────────────────────────────

HEAD_CSV   = CKPT_DIR / 'pruning_auc_comparison.csv'
TOKEN_CSV  = CKPT_DIR / 'token_pruning_auc.csv'
CA_CSV     = CKPT_DIR / 'ca_vs_standard_auc.csv'
CLIN_CSV   = CKPT_DIR / 'clinical_mlp_auc.csv'

series = [
    # Baseline
    load_col(HEAD_CSV,  'Label', 'baseline',     'baseline'),
    # Head pruning
    load_col(HEAD_CSV,  'Label', '25pct',         'head_25%'),
    load_col(HEAD_CSV,  'Label', '50pct',         'head_50%'),
    load_col(HEAD_CSV,  'Label', '75pct',         'head_75%'),
    # Standard DynamicViT
    load_col(TOKEN_CSV, 'Label', 'dynvit_70pct',  'std_keep70%'),
    load_col(TOKEN_CSV, 'Label', 'dynvit_50pct',  'std_keep50%'),
    load_col(TOKEN_CSV, 'Label', 'dynvit_30pct',  'std_keep30%'),
    # CA-DynamicViT
    load_col(CA_CSV,    'Label', 'ca_70',         'ca_keep70%'),
    load_col(CA_CSV,    'Label', 'ca_50',         'ca_keep50%'),
    load_col(CA_CSV,    'Label', 'ca_30',         'ca_keep30%'),
    # Clinical MLP
    load_col(CLIN_CSV,  'Label', 'clinical',      'clinical'),
]

# Late fusion — from current run
fusion_series = pd.Series(
    {l: fusion_auc.get(l, float('nan')) for l in LABEL_COLS},
    name='late_fusion')
series.append(fusion_series)

# ── build table ───────────────────────────────────────────────────────────────

df_all = pd.concat(series, axis=1).loc[rel]
df_all.index.name = 'Label'

# Add Val+ and Rare marker
meta = pd.DataFrame({
    'Val+': [int(val_pos.get(l, 0)) for l in rel],
    'Rare': ['★' if l in RARE_LABELS else '' for l in rel],
}, index=rel)

df_display = pd.concat([meta, df_all], axis=1)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print('FULL COMPARISON TABLE — AUC per label (reliable labels, Val+ ≥ 5)\n')
    print(df_display.round(4).to_string())

# ── macro AUC per method ──────────────────────────────────────────────────────

print('\nMacro AUC (reliable labels):')
for col in df_all.columns:
    vals = df_all[col].dropna().values
    if len(vals) == 0:
        print(f'  {col:<18}: —')
    else:
        print(f'  {col:<18}: {np.mean(vals):.4f}')

# Save
df_display.to_csv(CKPT_DIR / 'full_comparison_table.csv')
print('\nSaved full_comparison_table.csv')

In [ ]:
# ── Cell 10 — Plots ───────────────────────────────────────────────────────────

reliable = [l for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-label AUC — three main models
ax = axes[0]
x  = np.arange(len(reliable))
w  = 0.25
ax.bar(x - w, [df_res.loc[l, 'image_only']  for l in reliable], w,
       label='Image only', color='steelblue', alpha=0.85)
ax.bar(x,     [df_res.loc[l, 'clinical']    for l in reliable], w,
       label='Clinical only', color='coral', alpha=0.85)
ax.bar(x + w, [df_res.loc[l, 'late_fusion'] for l in reliable], w,
       label='Late fusion', color='seagreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(reliable, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('AUC')
ax.set_ylim(0.4, 1.05)
ax.set_title('AUC comparison: image vs clinical vs late fusion')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Right: ablation — macro AUC per configuration
ax = axes[1]
configs     = list(ablation_configs.keys())
macro_vals  = []
for cfg in configs:
    vals = [ablation_results[cfg].get(l, float('nan'))
            for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
    with warnings.catch_warnings(): warnings.simplefilter('ignore')
    macro_vals.append(float(np.nanmean(vals)))

colours = ['seagreen','coral','steelblue','orchid','steelblue','teal','coral']
ax.barh(configs, macro_vals, color=colours, alpha=0.8)
ax.axvline(macro_vals[0], linestyle='--', color='black', linewidth=0.8, label='Full fusion')
ax.set_xlabel('Macro AUC (reliable labels)')
ax.set_title('Ablation: contribution of each branch')
ax.set_xlim(0.4, max(macro_vals) + 0.05)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(MAPS_DIR / 'fusion_late_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved fusion_late_summary.png')